In [1]:
from qutip import *
import numpy as np
import scipy as sp
import math
import matplotlib.pyplot as plt
from scipy.special import genlaguerre as L
from scipy.optimize import root_scalar as root
from matplotlib import colormaps as cm
import matplotlib.ticker as ticker


In [26]:
fe3 = 642121496772645.10

In [2]:
def Omega(n,m,eta):
    return np.abs(np.exp(-eta**2/2.0) * eta**(np.abs(m)) * (math.factorial(n)/math.factorial(n+m))**(np.sign(m)/2.0) \
        * L(n,np.abs(m))(eta**2))             #returns unitless rabi rate of transition n -> n+m given Lamb-Dicke Parameter

def prob(n,nbar):
    return 1.0/(nbar+1)*(nbar/(nbar+1))**n    #returns prob of being in state n given nbar
    
eta = 0.06882
Ncut = 1000
omegas = np.zeros(Ncut)
#probs = np.zeros(Ncut)
#nbar = 10
for n in range(Ncut):
    omegas[n] = Omega(n,0,eta)
    #probs[n] = prob(n,nbar)

C:\Users\markl\AppData\Local\Programs\Python\Python313\Lib\site-packages\scipy\special\_orthogonal.py:670: RuntimeWarning: invalid value encountered in scalar divide
  hn = _gam(n + alpha + 1) / _gam(n + 1)
C:\Users\markl\AppData\Local\Programs\Python\Python313\Lib\site-packages\scipy\special\_orthogonal.py:133: RuntimeWarning: invalid value encountered in multiply
  np.poly1d.__init__(self, poly.coeffs * float(kn))
C:\Users\markl\AppData\Local\Programs\Python\Python313\Lib\site-packages\scipy\special\_orthogonal.py:119: RuntimeWarning: divide by zero encountered in scalar divide
  equiv_weights = [weights[k] / wfunc(roots[k]) for
C:\Users\markl\AppData\Local\Programs\Python\Python313\Lib\site-packages\scipy\special\_orthogonal.py:119: RuntimeWarning: invalid value encountered in scalar divide
  equiv_weights = [weights[k] / wfunc(roots[k]) for
C:\Users\markl\AppData\Local\Programs\Python\Python313\Lib\site-packages\scipy\special\_orthogonal.py:571: RuntimeWarning: overflow encountered

In [3]:
hbar = sp.constants.hbar
h = sp.constants.h
c = sp.constants.c
e = sp.constants.e
eVtoJ = 1.602176634e-19
amutokg = 1.66053906660e-27
nu0 = 642121496772645.10 #Hz E3 transition (Huntemann)

def expE(nbar, omega, Ncut):
    expE = 0.0
    for n in range(Ncut):
        expE += prob(n,nbar)*hbar*omega*(0.5+n)
    return expE

def dopAndStark(n): #gives instantaneous doppler and stark shift in angular frequency units
    Ncut = 1201
    omega = 2.0*np.pi*1.0e6 #Our trap
    Omegarf = 2.0*np.pi*16.026e6 #Our trap
    m = 170.936331517*amutokg
    alpha = 0.888e-40 #J m^2 V^-2 E3 transition (Lange)
    nu0 = 642121496772645.10 #Hz E3 transition (Huntemann)
    return -2.0*np.pi*nu0*hbar*omega*(0.5+n)*(3.0/(m*c**2) + (alpha/(h*nu0))*(3.0*m*Omegarf**2/(2.0*e**2)))
#from Lange eqns 2.11 and 2.12

Ncut = 2001
dopAndStarks = np.zeros(Ncut)
ns = np.linspace(0,Ncut-1,Ncut)
for ii in range(Ncut):
    dopAndStarks[ii] = dopAndStark(ns[ii])

In [4]:
def rhoee(psi, nbar):
    Ncut = np.shape(psi)[0]
    result = 0.
    for n in range(Ncut):
        result += prob(n,nbar)*(abs(psi[n,1]))**2.0
    return result

def rhogg(psi, nbar):
    Ncut = np.shape(psi)[0]
    result = 0.
    for n in range(Ncut):
        result += prob(n,nbar)*(abs(psi[n,0]))**2.0
    return result

def psi(g,e):
    return np.array([g, e])

def groundPsi(Ncut):
    result = np.zeros((Ncut,2),dtype="complex")
    result[:,0] = 1.
    return result

def subPulse(t, Omega0, delta, phi0, phi1):
    Omega = np.sqrt(Omega0**2 + delta**2)
    
    if Omega == 0.0:
        U = np.array([[1.0 , 0.0],
                      [0.0 , 1.0]])
        
    else:
        U = np.array([[np.cos(Omega*t/2.0) -(1.0j*delta/Omega)*np.sin(Omega*t/2.0), (1.0j*Omega0/Omega)*np.sin(Omega*t/2.0)],
                     [(1.0j*Omega0/Omega)*np.sin(Omega*t/2.0) , np.cos(Omega*t/2.0) + (1.0j*delta/Omega)*np.sin(Omega*t/2.0)]])
    
    V1 = np.array([[1.0 , 0.0],
                   [0.0 , np.exp(-1.0j*phi1)]])
    
    V0 = np.array([[1.0 , 0.0],
                   [0.0 , np.exp(1.0j*phi0)]])
    
    return np.exp(1.0j*delta*t/2.0)*V1@U@V0

def pulse(psi, t, Omega0, delta, phi0, phi1):
    Ncut = np.shape(psi)[0]
    result = np.zeros((Ncut,2),dtype="complex")
    if (Omega0==0):
        subpulse = subPulse(t, Omega0, delta, phi0, phi1)
        for n in range(Ncut):
            result[n,:] = subpulse@psi[n,:]
    else:
        for n in range(Ncut):
            result[n,:] = subPulse(t, Omega0*omegas[n], delta, phi0, phi1)@psi[n,:]
    return result

In [22]:
def RS(deld, psi0, t1, T, t2, del1, del2, Omega0, nbarf, Ncut, q):
    nbar0 = 0
    delp1 = del1
    delp2 = del2 + dopAndStark(nbarf) - dopAndStark(nbar0)
    phi = 0.5*(dopAndStark(nbarf)+dopAndStark(nbar0))*T
    #q = 0.9
    intState = subPulse(T, 0.0, deld, 0.0, phi)@subPulse(t1, q*Omega0, delp1, 0.0, 0.0)@psi0
    intStateP = subPulse(0.,0.,0.,0.,np.pi/2.)@intState 
    intStateM = subPulse(0.,0.,0.,0.,-np.pi/2.)@intState
    psiP = np.zeros((Ncut,2),dtype="complex")
    psiM = np.zeros((Ncut,2),dtype="complex")
    for n in range(Ncut):
        psiP[n,:] = intStateP
        psiM[n,:] = intStateM
    resultP = pulse(psiP, t2, q*Omega0, delp2, 0.0, 0.0)
    resultM = pulse(psiM, t2, q*Omega0, delp2, 0.0, 0.0)
    return rhoee(resultP,nbarf)-rhoee(resultM,nbarf)

In [5]:
def HRJumpFront(deld, psi0, t1, T, t2, del1, del2, Omega0, nbarf, Ncut, q):
    nbar0 = 0
    delp1 = del1
    delp2 = del2 + dopAndStark(nbarf) - dopAndStark(nbar0)
    phi = 0.5*(dopAndStark(nbarf)+dopAndStark(nbar0))*T
    #q = 0.9
    intState = subPulse(T, 0.0, deld, 0.0, phi)@subPulse(2.0*t1/3.0, -q*Omega0, delp1, 0.0, 0.0)@subPulse(t1/3.0, q*Omega0, delp1, 0.0, 0.0)@psi0
    intStateP = subPulse(0.,0.,0.,0.,np.pi/2.)@intState 
    intStateM = subPulse(0.,0.,0.,0.,-np.pi/2.)@intState
    psiP = np.zeros((Ncut,2),dtype="complex")
    psiM = np.zeros((Ncut,2),dtype="complex")
    for n in range(Ncut):
        psiP[n,:] = intStateP
        psiM[n,:] = intStateM
    resultP = pulse(psiP, t2, q*Omega0, delp2, 0.0, 0.0)
    resultM = pulse(psiM, t2, q*Omega0, delp2, 0.0, 0.0)
    return rhoee(resultP,nbarf)-rhoee(resultM,nbarf)

In [6]:
def shiftCurve(func, t1, T, t2, delp1, delp2, Omega0, nbarfs, Ncut, q):
    psi0 = psi(1.+0.j,0.+0.j)
    x0 = -0.5
    x1 = 0.5
    guessTol = 2.0
    tol = 1e-6
    pts = np.size(nbarfs)
    shifts = np.zeros(pts)
    for ii in range(pts):
        shifts[ii] = root(func, args=(psi0, t1, T, t2, delp1, delp2, Omega0, nbarfs[ii], Ncut, q), bracket=[x0,x1], xtol = tol).root
        print(shifts[ii]/(2.*np.pi))
        x0 = shifts[ii] - guessTol
        x1 = shifts[ii] + guessTol
        print([x0,x1])
    return shifts

In [27]:
psi0 = psi(1.+0.j,0.+0.j)
x0 = -0.5
x1 = 0.5
guessTol = 2.0
tol = 1e-6
t1 = 23.9e-3
t2 = 23.9e-3
T = 150e-3
Omega0 = np.pi/(2.*23.9e-3)
nbarf = 183*(t1+T+t2)
q=1
Ncut = 500
1e18*root(RS, args=(psi0, t1, T, t2, 0, 0, Omega0, nbarf, Ncut, q), bracket=[x0,x1], xtol = tol).root/(2.0*np.pi*fe3)

-1.5339706879613528

In [7]:
delPts = 7
delPrimes = 2.0*np.pi*np.linspace(0.0, 0.6, delPts)
#delPrimes= np.array([0])
nbarfPts = 25
nbarfs = np.linspace(0, 60, nbarfPts)
resultslo = np.zeros([delPts, nbarfPts])

t1 = 3*23.9e-3
t2 = 23.9e-3
T = 150e-3
Omega0 = np.pi/(2.*23.9e-3)
#delPrime = 1. * 2.0*np.pi
Ncut = 500
qlo = 0.99

for ii in range(delPts):
    resultslo[ii,:] = shiftCurve(HRJumpFront, t1, T, t2, delPrimes[ii], delPrimes[ii], Omega0, nbarfs, Ncut, qlo)

-3.2502912742762103e-05
[np.float64(-2.0002042218237857), np.float64(1.9997957781762141)]
-9.746669925280525e-05
[np.float64(-2.0006124013326847), np.float64(1.9993875986673155)]
-0.00016261193744266256
[np.float64(-2.001021720936112), np.float64(1.9989782790638884)]
-0.00022793108323476046
[np.float64(-2.00143213323323), np.float64(1.99856786676677)]
-0.0002934073070230909
[np.float64(-2.0018435324805064), np.float64(1.9981564675194934)]
-0.0003590253430964122
[np.float64(-2.0022558227606484), np.float64(1.9977441772393516)]
-0.0004247713598616184
[np.float64(-2.002668917167193), np.float64(1.9973310828328068)]
-0.0004906328374770644
[np.float64(-2.0030827370356556), np.float64(1.9969172629643444)]
-0.0005565984509415843
[np.float64(-2.003497211208955), np.float64(1.996502788791045)]
-0.0006226579613408385
[np.float64(-2.003912275354095), np.float64(1.9960877246459048)]
-0.0006888021176364885
[np.float64(-2.0043278713450876), np.float64(1.9956721286549122)]
-0.0007550225703016338
[np.

In [8]:
delPts = 7
delPrimes = 2.0*np.pi*np.linspace(0.0, 0.6, delPts)
#delPrimes= np.array([0])
nbarfPts = 25
nbarfs = np.linspace(0, 60, nbarfPts)
resultshi = np.zeros([delPts, nbarfPts])

t1 = 3*23.9e-3
t2 = 23.9e-3
T = 150e-3
Omega0 = np.pi/(2.*23.9e-3)
#delPrime = 1. * 2.0*np.pi
Ncut = 500
qhi = 1.01

for ii in range(delPts):
    resultshi[ii,:] = shiftCurve(HRJumpFront, t1, T, t2, delPrimes[ii], delPrimes[ii], Omega0, nbarfs, Ncut, qhi)

-3.250291274270909e-05
[np.float64(-2.0002042218237857), np.float64(1.9997957781762146)]
-9.728701639699347e-05
[np.float64(-2.000611272352005), np.float64(1.9993887276479951)]
-0.00016226570467803877
[np.float64(-2.0010195454914923), np.float64(1.9989804545085077)]
-0.00022742994419195392
[np.float64(-2.0014289844837596), np.float64(1.9985710155162404)]
-0.00029276164628833345
[np.float64(-2.0018394756744646), np.float64(1.9981605243255354)]
-0.0003582444165190927
[np.float64(-2.002250916054252), np.float64(1.9977490839457481)]
-0.00042386341354607533
[np.float64(-2.002663212372244), np.float64(1.9973367876277563)]
-0.0004896052155871216
[np.float64(-2.0030762802968956), np.float64(1.9969237197031044)]
-0.0005554576927917767
[np.float64(-2.003490043614109), np.float64(1.9965099563858908)]
-0.0006214098885867871
[np.float64(-2.0039044334817047), np.float64(1.9960955665182953)]
-0.0006874519124748162
[np.float64(-2.0043193877558543), np.float64(1.9956806122441457)]
-0.000753574845757979

In [9]:
delPts = 7
delPrimes = 2.0*np.pi*np.linspace(0.0, 0.6, delPts)
#delPrimes= np.array([0])
nbarfPts = 25
nbarfs = np.linspace(0, 60, nbarfPts)
results = np.zeros([delPts, nbarfPts])

t1 = 3*23.9e-3
t2 = 23.9e-3
T = 150e-3
Omega0 = np.pi/(2.*23.9e-3)
#delPrime = 1. * 2.0*np.pi
Ncut = 500
q = 1.0

for ii in range(delPts):
    results[ii,:] = shiftCurve(HRJumpFront, t1, T, t2, delPrimes[ii], delPrimes[ii], Omega0, nbarfs, Ncut, q)

-3.250291274279744e-05
[np.float64(-2.000204221823786), np.float64(1.999795778176214)]
-9.737787917598828e-05
[np.float64(-2.0006118432596827), np.float64(1.9993881567403171)]
-0.00016244075543074202
[np.float64(-2.0010206453678094), np.float64(1.9989793546321903)]
-0.00022768326701139515
[np.float64(-2.0014305761579765), np.float64(1.9985694238420233)]
-0.0002930879668685876
[np.float64(-2.00184152600714), np.float64(1.9981584739928602)]
-0.00035863903552530764
[np.float64(-2.0022533955185935), np.float64(1.9977466044814063)]
-0.0004243221457850038
[np.float64(-2.0026660946719073), np.float64(1.9973339053280927)]
-0.0004901243349333139
[np.float64(-2.003079542019944), np.float64(1.996920457980056)]
-0.0005560338826104023
[np.float64(-2.0034936639215117), np.float64(1.9965063360784883)]
-0.0006220401972203338
[np.float64(-2.00390839382765), np.float64(1.9960916061723502)]
-0.000688133713352373
[np.float64(-2.0043236716371107), np.float64(1.9956763283628896)]
-0.0007543058015484357
[np.

In [10]:
#delPrime1 = 2.0*np.pi*0.186
delPrime1 = 2.0*np.pi*0.191
delPrime2 = 2.0*np.pi*0.365

optResult1lo = shiftCurve(HRJumpFront, t1, T, t2, delPrime1, delPrime1, Omega0, nbarfs, Ncut, qlo)
optResult2lo = shiftCurve(HRJumpFront, t1, T, t2, delPrime2, delPrime2, Omega0, nbarfs, Ncut, qlo)

optResult1hi = shiftCurve(HRJumpFront, t1, T, t2, delPrime1, delPrime1, Omega0, nbarfs, Ncut, qhi)
optResult2hi = shiftCurve(HRJumpFront, t1, T, t2, delPrime2, delPrime2, Omega0, nbarfs, Ncut, qhi)

optResult1 = shiftCurve(HRJumpFront, t1, T, t2, delPrime1, delPrime1, Omega0, nbarfs, Ncut, 1.0)
optResult2 = shiftCurve(HRJumpFront, t1, T, t2, delPrime2, delPrime2, Omega0, nbarfs, Ncut, 1.0)

-3.3859716732642185e-05
[np.float64(-2.0002127468746798), np.float64(1.9997872531253202)]
2.2129914567920092e-05
[np.float64(-1.9998609536459377), np.float64(2.0001390463540623)]
7.010606137102035e-05
[np.float64(-1.9995595106252493), np.float64(2.0004404893747507)]
0.00011065499139287604
[np.float64(-1.9993047341839143), np.float64(2.000695265816086)]
0.00014428790454349777
[np.float64(-1.9990934123581685), np.float64(2.0009065876418313)]
0.00017148551045593374
[np.float64(-1.998922524760309), np.float64(2.0010774752396907)]
0.0001926978134781366
[np.float64(-1.9987892439296286), np.float64(2.0012107560703716)]
0.00020834459348397272
[np.float64(-1.9986909323113913), np.float64(2.001309067688609)]
0.00021881641238274667
[np.float64(-1.998625135932747), np.float64(2.001374864067253)]
0.00022447608431733337
[np.float64(-1.998589575165204), np.float64(2.0014104248347957)]
0.00022562428494332618
[np.float64(-1.9985823608079012), np.float64(2.001417639192099)]
0.0002226831028115136
[np.flo

In [11]:
optResult1[18]/(2*np.pi)/nu0*1e18

np.float64(0.23781309690996652)

In [12]:
nbarfs[18]

np.float64(45.0)

In [13]:
def shiftToFrac(shift):
    return shift/nu0*1e18

def fracToShift(frac):
    return frac*nu0/1e18

In [14]:
for ii in range(2,4):
    print(ii)

2
3


In [15]:
cm=plt.get_cmap('plasma',delPts+1)

fig,ax1=plt.subplots(figsize=(14,10), dpi=700)
#plt.figure(figsize=(14,8), dpi=500)
ax1.set_xlabel(r"Motional State $\bar{n}_F$",fontsize="35")
#ax1.set_ylabel(r"Clock Shift (mHz)",fontsize="35")
ax1.set_ylabel(r"Clock Error (mHz)",fontsize="35")
y1, y2 = -2.5,2.5
ax1.set_ylim(y1,y2)
plt.xlim([0,70])
ax1.tick_params(axis="x", labelsize=25)
ax1.grid(axis='y', which='major', linewidth=0.7)
ax1.grid(axis='x', which='major', linewidth=0.7)
ax1.tick_params(axis="y", labelsize="25")
ax2 = ax1.twinx()
#ax2.set_ylabel(r"$\Delta \nu/\nu_{E3}$",fontsize="35")
ax2.set_ylabel(r"Fractional Frequency Error",fontsize="35")
ax2.tick_params(axis="y", labelsize="25")
ax2.set_ylim(y1*1e-3/642121496772645.13, y2*1e-3/642121496772645.13)
ax2.yaxis.offsetText.set_fontsize(20)

formatter = ticker.ScalarFormatter(useMathText=True)
formatter.set_powerlimits((0, 0))   # Always use scientific notation
ax2.yaxis.set_major_formatter(formatter)
plt.draw()  # Ensures the offset text is updated

ax1.plot(nbarfs, 1e3*results[5,:]/(2.0*np.pi), label=r"$\delta = $" + str(round(delPrimes[5]/(2.0*np.pi), 2)), color=cm(5),linewidth=3)
ax2.plot(nbarfs, results[5,:]/(2.0*np.pi*642121496772645.13), label=r"$\delta = $" + str(round(delPrimes[5]/(2.0*np.pi), 2)), color=cm(delPts-1-5),linewidth=3)
ax1.fill_between(nbarfs, 1e3*resultslo[5,:]/(2.0*np.pi), 1e3*resultshi[5,:]/(2.0*np.pi), color=cm(delPts-1-5), alpha=0.1)

ax1.plot(nbarfs, 1e3*results[4,:]/(2.0*np.pi), label=r"$\delta = $" + str(round(delPrimes[4]/(2.0*np.pi), 2)), color=cm(4),linewidth=3)
ax2.plot(nbarfs, results[4,:]/(2.0*np.pi*642121496772645.13), label=r"$\delta = $" + str(round(delPrimes[4]/(2.0*np.pi), 2)), color=cm(delPts-1-4),linewidth=3)
ax1.fill_between(nbarfs, 1e3*resultslo[4,:]/(2.0*np.pi), 1e3*resultshi[4,:]/(2.0*np.pi), color=cm(delPts-1-4), alpha=0.1)

ax1.plot(nbarfs, 1e3*optResult2/(2.0*np.pi), label=r"$\delta = $" + str(round(delPrime2/(2*np.pi),3)), color='royalblue',linewidth=4)
ax2.plot(nbarfs, optResult2/(2.0*np.pi*642121496772645.13), label=r"$\delta = $" + str(round(delPrime2/(2*np.pi),3)), color='royalblue',linewidth=4)

ax1.plot(nbarfs, 1e3*results[3,:]/(2.0*np.pi), label=r"$\delta = $" + str(round(delPrimes[3]/(2.0*np.pi), 2)), color=cm(3),linewidth=3)
ax2.plot(nbarfs, results[3,:]/(2.0*np.pi*642121496772645.13), label=r"$\delta = $" + str(round(delPrimes[3]/(2.0*np.pi), 2)), color=cm(delPts-1-3),linewidth=3)
ax1.fill_between(nbarfs, 1e3*resultslo[3,:]/(2.0*np.pi), 1e3*resultshi[3,:]/(2.0*np.pi), color=cm(delPts-1-3), alpha=0.1)

ax1.plot(nbarfs, 1e3*results[2,:]/(2.0*np.pi), label=r"$\delta = $" + str(round(delPrimes[2]/(2.0*np.pi), 2)), color=cm(2),linewidth=3)
ax2.plot(nbarfs, results[2,:]/(2.0*np.pi*642121496772645.13), label=r"$\delta = $" + str(round(delPrimes[2]/(2.0*np.pi), 2)), color=cm(delPts-1-2),linewidth=3)
ax1.fill_between(nbarfs, 1e3*resultslo[2,:]/(2.0*np.pi), 1e3*resultshi[2,:]/(2.0*np.pi), color=cm(delPts-1-2), alpha=0.1)

ax1.plot(nbarfs, 1e3*optResult1/(2.0*np.pi), label=r"$\delta = $" + str(round(delPrime1/(2*np.pi),3)), color='limegreen',linewidth=4)
ax2.plot(nbarfs, optResult1/(2.0*np.pi*642121496772645.13), label=r"$\delta = $" + str(round(delPrime1/(2*np.pi),3)), color='limegreen',linewidth=4)
ax1.fill_between(nbarfs, 1e3*optResult1lo/(2.0*np.pi), 1e3*optResult1hi/(2.0*np.pi), color='limegreen', alpha=0.1)

ax1.plot(nbarfs, 1e3*results[1,:]/(2.0*np.pi), label=r"$\delta = $" + str(round(delPrimes[1]/(2.0*np.pi), 2)), color=cm(1),linewidth=3)
ax2.plot(nbarfs, results[1,:]/(2.0*np.pi*642121496772645.13), label=r"$\delta = $" + str(round(delPrimes[1]/(2.0*np.pi), 2)), color=cm(delPts-1-1),linewidth=3)
ax1.fill_between(nbarfs, 1e3*resultslo[1,:]/(2.0*np.pi), 1e3*resultshi[1,:]/(2.0*np.pi), color=cm(delPts-1-1), alpha=0.1)

ax1.plot(nbarfs, 1e3*results[0,:]/(2.0*np.pi), label=r"$\delta = $" + str(round(delPrimes[0]/(2.0*np.pi), 2)), color=cm(0),linewidth=3)
ax2.plot(nbarfs, results[0,:]/(2.0*np.pi*642121496772645.13), label=r"$\delta = $" + str(round(delPrimes[0]/(2.0*np.pi), 2)), color=cm(delPts-1-0),linewidth=3)
ax1.fill_between(nbarfs, 1e3*resultslo[0,:]/(2.0*np.pi), 1e3*resultshi[0,:]/(2.0*np.pi), color=cm(delPts-1-0), alpha=0.1)

#plt.title("HRS 180 3+1",fontsize="40")
plt.legend(ncol=2, fontsize="20", title=r"Pulse Detuning $\delta/(2\pi)$ (mHz)", title_fontsize="20")#, bbox_to_anchor = (1.45,0.5), loc='center right')
#plt.tight_layout()
plt.set_cmap("viridis")
#plt.ylim([-1.3,0.9])
plt.savefig("colorBlock.pdf")
plt.show()

In [16]:
cm=plt.get_cmap('plasma',delPts+1)

fig,ax1=plt.subplots(figsize=(14,10), dpi=700)
#plt.figure(figsize=(14,8), dpi=500)
ax1.set_xlabel(r"Motional State $\bar{n}_{final}$",fontsize="35")
#ax1.set_ylabel(r"Clock Shift (mHz)",fontsize="35")
ax1.set_ylabel(r"Clock Error (mHz)",fontsize="35")
y1, y2 = -2.5,2.5
ax1.set_ylim(y1,y2)
plt.xlim([0,70])
ax1.tick_params(axis="x", labelsize=25)
ax1.grid(axis='y', which='major', linewidth=0.7)
ax1.grid(axis='x', which='major', linewidth=0.7)
ax1.tick_params(axis="y", labelsize="25")
ax2 = ax1.twinx()
#ax2.set_ylabel(r"$\Delta \nu/\nu_{E3}$",fontsize="35")
ax2.set_ylabel(r"Fractional Error $\times 10^{-18}$",fontsize="35")
ax2.tick_params(axis="y", labelsize="25")
ax2.set_ylim(1e18*y1*1e-3/642121496772645.13, 1e18*y2*1e-3/642121496772645.13)
ax2.yaxis.offsetText.set_fontsize(20)

#formatter = ticker.ScalarFormatter(useMathText=True)
#formatter.set_powerlimits((0, 0))   # Always use scientific notation
#ax2.yaxis.set_major_formatter(formatter)
#plt.draw()  # Ensures the offset text is updated
#ax2.ticklabel_format(axis='y', style='plain', useOffset=False)

ax1.plot(nbarfs, 1e3*results[5,:]/(2.0*np.pi), label=r"$\delta = 500$", color=cm(5),linewidth=3)
ax2.plot(nbarfs, 1e18*results[5,:]/(2.0*np.pi*642121496772645.13), label=r"$\delta = 500$", color=cm(delPts-1-5),linewidth=3)
ax1.fill_between(nbarfs, 1e3*resultslo[5,:]/(2.0*np.pi), 1e3*resultshi[5,:]/(2.0*np.pi), color=cm(delPts-1-5), alpha=0.1)

ax1.plot(nbarfs, 1e3*results[4,:]/(2.0*np.pi), label=r"$\delta = 400$", color=cm(4),linewidth=3)
ax2.plot(nbarfs, 1e18*results[4,:]/(2.0*np.pi*642121496772645.13), label=r"$\delta = 400$", color=cm(delPts-1-4),linewidth=3)
ax1.fill_between(nbarfs, 1e3*resultslo[4,:]/(2.0*np.pi), 1e3*resultshi[4,:]/(2.0*np.pi), color=cm(delPts-1-4), alpha=0.1)

#ax1.plot(nbarfs, 1e3*optResult2/(2.0*np.pi), label=r"$\delta = 365$", color='royalblue',linewidth=4)
#ax2.plot(nbarfs, optResult2/(2.0*np.pi*642121496772645.13), label=r"$\delta = 365$", color='royalblue',linewidth=4)

ax1.plot(nbarfs, 1e3*results[3,:]/(2.0*np.pi), label=r"$\delta = 300$", color=cm(3),linewidth=3)
ax2.plot(nbarfs, 1e18*results[3,:]/(2.0*np.pi*642121496772645.13), label=r"$\delta = 300$", color=cm(delPts-1-3),linewidth=3)
ax1.fill_between(nbarfs, 1e3*resultslo[3,:]/(2.0*np.pi), 1e3*resultshi[3,:]/(2.0*np.pi), color=cm(delPts-1-3), alpha=0.1)

#ax1.plot(nbarfs, 1e3*results[2,:]/(2.0*np.pi), label=r"$\delta = 200$" , color=cm(2),linewidth=3)
#ax2.plot(nbarfs, results[2,:]/(2.0*np.pi*642121496772645.13), label=r"$\delta = 200$" , color=cm(delPts-1-2),linewidth=3)
#ax1.fill_between(nbarfs, 1e3*resultslo[2,:]/(2.0*np.pi), 1e3*resultshi[2,:]/(2.0*np.pi), color=cm(delPts-1-2), alpha=0.1)

ax1.plot(nbarfs, 1e3*optResult1/(2.0*np.pi), label=r"$\delta = 191$", color='limegreen',linewidth=4)
ax2.plot(nbarfs, 1e18*optResult1/(2.0*np.pi*642121496772645.13), label=r"$\delta = 186$", color='limegreen',linewidth=4)
ax1.fill_between(nbarfs, 1e3*optResult1lo/(2.0*np.pi), 1e3*optResult1hi/(2.0*np.pi), color='limegreen', alpha=0.2)

ax1.axhline(y=1e3*np.max(optResult1)/(2.0*np.pi), color='limegreen', linestyle='--', linewidth=2)
ax1.axhline(y=1e3*np.min(optResult1)/(2.0*np.pi), color='limegreen', linestyle='--', linewidth=2)

ax1.plot(nbarfs, 1e3*results[1,:]/(2.0*np.pi), label=r"$\delta = 100$", color=cm(1),linewidth=3)
ax2.plot(nbarfs, 1e18*results[1,:]/(2.0*np.pi*642121496772645.13), label=r"$\delta = 100$" , color=cm(delPts-1-1),linewidth=3)
ax1.fill_between(nbarfs, 1e3*resultslo[1,:]/(2.0*np.pi), 1e3*resultshi[1,:]/(2.0*np.pi), color=cm(delPts-1-1), alpha=0.1)

ax1.plot(nbarfs, 1e3*results[0,:]/(2.0*np.pi), label=r"$\delta = 0$" , color=cm(0),linewidth=3)
ax2.plot(nbarfs, 1e18*results[0,:]/(2.0*np.pi*642121496772645.13), label=r"$\delta = 0$" , color=cm(delPts-1-0),linewidth=3)
ax1.fill_between(nbarfs, 1e3*resultslo[0,:]/(2.0*np.pi), 1e3*resultshi[0,:]/(2.0*np.pi), color=cm(delPts-1-0), alpha=0.1)

#plt.title("HRS 180 3+1",fontsize="40")
plt.legend(ncol=2, fontsize="20", title=r"Pulse Detuning $\delta/(2\pi)$ (mHz)", title_fontsize="20")#, bbox_to_anchor = (1.45,0.5), loc='center right')
#plt.tight_layout()
plt.set_cmap("viridis")
#plt.ylim([-1.3,0.9])
plt.savefig("colorBlock.pdf")
plt.show()

In [29]:
cm=plt.get_cmap('plasma',delPts+1)

fig,ax1=plt.subplots(figsize=(14,10), dpi=700)
#plt.figure(figsize=(14,8), dpi=500)
ax1.set_xlabel(r"Motional State $\bar{n}_{final}$",fontsize="35")
#ax1.set_ylabel(r"Clock Shift (mHz)",fontsize="35")
ax1.set_ylabel(r"Clock Error (mHz)",fontsize="35")
y1, y2 = -2.5,2.5
ax1.set_ylim(y1,y2)
plt.xlim([0,60])
ax1.tick_params(axis="x", labelsize=25)
ax1.grid(axis='y', which='major', linewidth=0.7)
ax1.grid(axis='x', which='major', linewidth=0.7)
ax1.tick_params(axis="y", labelsize="25")
ax2 = ax1.twinx()
#ax2.set_ylabel(r"$\Delta \nu/\nu_{E3}$",fontsize="35")
ax2.set_ylabel(r"Fractional Error $\times 10^{-18}$",fontsize="35")
ax2.tick_params(axis="y", labelsize="25")
ax2.set_ylim(1e18*y1*1e-3/642121496772645.13, 1e18*y2*1e-3/642121496772645.13)
ax2.yaxis.offsetText.set_fontsize(20)

#formatter = ticker.ScalarFormatter(useMathText=True)
#formatter.set_powerlimits((0, 0))   # Always use scientific notation
#ax2.yaxis.set_major_formatter(formatter)
#plt.draw()  # Ensures the offset text is updated
#ax2.ticklabel_format(axis='y', style='plain', useOffset=False)

ax1.plot(nbarfs, 1e3*results[5,:]/(2.0*np.pi), label=r"$500$", color=cm(5),linewidth=3)
ax2.plot(nbarfs, 1e18*results[5,:]/(2.0*np.pi*642121496772645.13), label=r"$500$", color=cm(delPts-1-5),linewidth=3)
ax1.plot(nbarfs, 1e3*resultslo[5,:]/(2.0*np.pi), color=cm(delPts-1-5), ls=":")
ax1.plot(nbarfs, 1e3*resultshi[5,:]/(2.0*np.pi), color=cm(delPts-1-5), ls=":")

ax1.plot(nbarfs, 1e3*results[4,:]/(2.0*np.pi), label=r"$400$", color=cm(4),linewidth=3)
ax2.plot(nbarfs, 1e18*results[4,:]/(2.0*np.pi*642121496772645.13), label=r"$400$", color=cm(delPts-1-4),linewidth=3)
ax1.plot(nbarfs, 1e3*resultslo[4,:]/(2.0*np.pi), color=cm(delPts-1-4), ls=":")
ax1.plot(nbarfs, 1e3*resultshi[4,:]/(2.0*np.pi), color=cm(delPts-1-4), ls=":")

#ax1.plot(nbarfs, 1e3*optResult2/(2.0*np.pi), label=r"$365$", color='royalblue',linewidth=4)
#ax2.plot(nbarfs, optResult2/(2.0*np.pi*642121496772645.13), label=r"$365$", color='royalblue',linewidth=4)

ax1.plot(nbarfs, 1e3*results[3,:]/(2.0*np.pi), label=r"$300$", color=cm(3),linewidth=3)
ax2.plot(nbarfs, 1e18*results[3,:]/(2.0*np.pi*642121496772645.13), label=r"$300$", color=cm(delPts-1-3),linewidth=3)
ax1.plot(nbarfs, 1e3*resultslo[3,:]/(2.0*np.pi), color=cm(delPts-1-3), ls=":")
ax1.plot(nbarfs, 1e3*resultshi[3,:]/(2.0*np.pi), color=cm(delPts-1-3), ls=":")

#ax1.plot(nbarfs, 1e3*results[2,:]/(2.0*np.pi), label=r"$200$" , color=cm(2),linewidth=3)
#ax2.plot(nbarfs, results[2,:]/(2.0*np.pi*642121496772645.13), label=r"$200$" , color=cm(delPts-1-2),linewidth=3)
#ax1.fill_between(nbarfs, 1e3*resultslo[2,:]/(2.0*np.pi), 1e3*resultshi[2,:]/(2.0*np.pi), color=cm(delPts-1-2), alpha=0.1)

ax1.plot(nbarfs, 1e3*optResult1/(2.0*np.pi), label=r"$191$", color='limegreen',linewidth=4)
ax2.plot(nbarfs, 1e18*optResult1/(2.0*np.pi*642121496772645.13), label=r"$191$", color='limegreen',linewidth=4)
ax1.plot(nbarfs, 1e3*optResult1lo/(2.0*np.pi), color="limegreen", ls=":")
ax1.plot(nbarfs, 1e3*optResult1hi/(2.0*np.pi), color="limegreen", ls=":")
ax1.fill_between(nbarfs, 1e3*np.min(optResult1lo)/(2.0*np.pi), 1e3*np.max(optResult1hi)/(2.0*np.pi), color='limegreen', alpha=0.2)

#ax1.axhline(y=1e3*np.max(optResult1)/(2.0*np.pi), color='limegreen', linestyle=':', linewidth=2)
#ax1.axhline(y=1e3*np.min(optResult1)/(2.0*np.pi), color='limegreen', linestyle=':', linewidth=2)

ax1.plot(nbarfs, 1e3*results[1,:]/(2.0*np.pi), label=r"$100$", color=cm(1),linewidth=3)
ax2.plot(nbarfs, 1e18*results[1,:]/(2.0*np.pi*642121496772645.13), label=r"$100$" , color=cm(delPts-1-1),linewidth=3)
ax1.plot(nbarfs, 1e3*resultslo[1,:]/(2.0*np.pi), color=cm(delPts-1-1), ls=":")
ax1.plot(nbarfs, 1e3*resultshi[1,:]/(2.0*np.pi), color=cm(delPts-1-1), ls=":")

ax1.plot(nbarfs, 1e3*results[0,:]/(2.0*np.pi), label=r"$0$" , color=cm(0),linewidth=3)
ax2.plot(nbarfs, 1e18*results[0,:]/(2.0*np.pi*642121496772645.13), label=r"$0$" , color=cm(delPts-1-0),linewidth=3)
ax1.plot(nbarfs, 1e3*resultslo[0,:]/(2.0*np.pi), color=cm(delPts-1-0), ls=":")
ax1.plot(nbarfs, 1e3*resultshi[0,:]/(2.0*np.pi), color=cm(delPts-1-0), ls=":")
ax1.fill_between(nbarfs, 1e3*np.min(resultslo[0,:])/(2.0*np.pi), 1e3*np.max(resultshi[0,:])/(2.0*np.pi), color=cm(delPts-1-0), alpha=0.2)


#plt.title("HRS 180 3+1",fontsize="40")
plt.legend(ncol=2, fontsize="24", title=r"Pulse Detuning $\delta'/(2\pi)$ (mHz)", title_fontsize="24")#, bbox_to_anchor = (1.45,0.5), loc='center right')
#plt.tight_layout()
plt.set_cmap("viridis")
#plt.ylim([-1.3,0.9])
plt.savefig("colorBlock.pdf", bbox_inches="tight")
plt.show()

In [21]:
1e18*(np.min(resultslo[0,:])-np.max(resultshi[0,:]))/(2.0*np.pi*642121496772645.13)/2

np.float64(-1.236352317174649)

In [19]:
1e18*(np.min(optResult1lo)-np.max(optResult1hi))/(2.*np.pi*642121496772645.13)/2

np.float64(-0.29350033173752366)

In [122]:
nbarfs[:23]

array([ 0. ,  2.5,  5. ,  7.5, 10. , 12.5, 15. , 17.5, 20. , 22.5, 25. ,
       27.5, 30. , 32.5, 35. , 37.5, 40. , 42.5, 45. , 47.5, 50. , 52.5,
       55. ])

In [134]:
np.sqrt(16-3.2**2/2)

np.float64(3.298484500494128)

In [133]:
np.max(resultshi[0,:])-np.min(resultslo[0,:])

np.float64(0.009976295866401412)

In [128]:
1e18*np.max(optResult1hi)/(2.0*np.pi*642121496772645.13)-1e18*np.min(optResult1lo)/(2.0*np.pi*642121496772645.13)

np.float64(0.5870006634750473)

In [129]:
1e18*np.max(resultshi[0,:])/(2.0*np.pi*642121496772645.13)-1e18*np.min(resultslo[0,:])/(2.0*np.pi*642121496772645.13)

np.float64(2.472704634349298)

In [130]:
2.472704634349298/0.5870006634750473

4.2124392495757546

In [100]:
1e18*np.max(optResult1hi)/(2.0*np.pi*642121496772645.13)-1e18*np.min(optResult1lo)/(2.0*np.pi*642121496772645.13)

np.float64(0.8609263256198196)

In [101]:
1e18*np.max(resultshi[0,:])/(2.0*np.pi*642121496772645.13)-1e18*np.min(resultslo[0,:])/(2.0*np.pi*642121496772645.13)

np.float64(2.888822910178257)

In [102]:
2.888822910178257/0.8609263256198196

3.3554821408190336